# 🌍 Notebook 3: Real-World Retry Patterns

Notebooks 1 and 2 showed *how* to time retries. Real systems need more than
that. This notebook covers the patterns you actually ship to production:

1. 🚦 **Classify errors** — retryable vs terminal (don't retry a `400`).
2. ⏱️ **Honor `Retry-After`** — if the server tells you when to come back, listen.
3. ⌛ **Overall deadline** — stop retrying once the caller has given up.
4. 💰 **Retry budget** — protect downstream under global failure.
5. 🔑 **Idempotency key** — make `POST` safe to retry.

We use **only the Python standard library** — no new dependencies. The HTTP
interactions are simulated with a tiny fake-client so this notebook runs
offline, deterministically.

## 🛠️ Setup

```bash
cd 05-microservices/retry
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## A fake HTTP client

In real code you'd use `requests`, `httpx`, or `urllib`. To keep this lab
self-contained we simulate responses with a small helper. The behavior you'd
see from a real server is identical.

In [ ]:
import random, time, uuid
from dataclasses import dataclass, field

@dataclass
class Response:
    status: int
    body: str = ''
    headers: dict = field(default_factory=dict)

class ConnectionError(Exception):
    """Network went away (DNS, TCP reset, TLS — truly transient)."""

class FakeServer:
    """Deterministic, scripted server for the demos."""
    def __init__(self, script):
        # script is a list of outcomes: 200, 500, 503, 'retry-after:2', 'conn-drop', 400, ...
        self.script = list(script)
        self.i = 0
        self.seen_keys = set()

    def post(self, path, idempotency_key=None):
        # Deduplicate by idempotency key → real servers do this
        if idempotency_key and idempotency_key in self.seen_keys:
            return Response(200, 'replayed-from-cache')
        outcome = self.script[min(self.i, len(self.script) - 1)]
        self.i += 1
        if outcome == 'conn-drop':
            raise ConnectionError('tcp reset')
        if isinstance(outcome, str) and outcome.startswith('retry-after:'):
            secs = int(outcome.split(':')[1])
            return Response(503, 'busy', {'Retry-After': str(secs)})
        if outcome == 200 and idempotency_key:
            self.seen_keys.add(idempotency_key)
        return Response(outcome, 'ok' if outcome == 200 else f'err {outcome}')


## 1. 🚦 Classify errors — don't retry a `400`

**Retryable**: network errors, `5xx` (except `501 Not Implemented`), `429 Too Many Requests`.

**Terminal**: most `4xx` (`400 Bad Request`, `401 Unauthorized`, `403 Forbidden`,
`404 Not Found`, `422 Unprocessable Entity`). The input is wrong — retrying
won't fix that, it just wastes time and money.

Failing to classify is one of the most common real-world retry bugs.

In [ ]:
RETRYABLE_STATUS = {408, 425, 429, 500, 502, 503, 504}

def is_retryable(exc_or_resp):
    if isinstance(exc_or_resp, ConnectionError):
        return True
    if isinstance(exc_or_resp, Response):
        return exc_or_resp.status in RETRYABLE_STATUS
    return False

# Quick sanity check:
print(is_retryable(Response(200)))              # False — success, no retry needed
print(is_retryable(Response(400)))              # False — caller error
print(is_retryable(Response(503)))              # True  — server unavailable
print(is_retryable(ConnectionError('reset')))   # True  — transient


## 2. ⏱️ + ⌛ Retry with `Retry-After`, deadline, and jitter

This is the real workhorse. It combines everything we've learned so far:

In [ ]:
def backoff_delay(attempt, base=0.1, factor=2.0, cap=2.0):
    """Full-jitter exponential backoff, capped."""
    exp = min(cap, base * (factor ** (attempt - 1)))
    return random.uniform(0, exp)

def parse_retry_after(value):
    """`Retry-After` is either delta-seconds OR an HTTP-date (RFC 9110 §10.2.3).

    Blindly calling float() on it is a real, common bug: a date header raises
    ValueError and the retry loop dies. Handle both, and ignore garbage."""
    if value is None:
        return None
    try:
        return max(0.0, float(value))            # delta-seconds form
    except ValueError:
        pass
    from email.utils import parsedate_to_datetime
    from datetime import datetime, timezone
    try:
        when = parsedate_to_datetime(value)      # HTTP-date form
    except (TypeError, ValueError):
        return None                              # malformed → fall back to our backoff
    if when.tzinfo is None:
        when = when.replace(tzinfo=timezone.utc)
    return max(0.0, (when - datetime.now(timezone.utc)).total_seconds())

def call_with_retry(do_call, *, max_attempts=5, deadline_s=3.0):
    """Call `do_call()` with retries.

    - Stops at max_attempts OR when the overall deadline is hit.
    - Honors `Retry-After` on 429/503 (seconds *or* HTTP-date).
    - Does NOT retry terminal errors (4xx).
    - Never sleeps after the final attempt.
    """
    start = time.monotonic()
    for attempt in range(1, max_attempts + 1):
        remaining = deadline_s - (time.monotonic() - start)
        if remaining <= 0:
            raise TimeoutError(f'deadline exceeded after {attempt - 1} attempts')
        try:
            resp = do_call()
        except ConnectionError as e:
            if attempt == max_attempts:
                raise
            sleep = min(backoff_delay(attempt), remaining)
            print(f'  attempt {attempt}: {e} → sleep {sleep:.2f}s')
            time.sleep(sleep)
            continue

        if resp.status == 200:
            return resp, attempt
        if not is_retryable(resp):
            raise RuntimeError(f'terminal {resp.status}: {resp.body}')
        if attempt == max_attempts:
            break                                # out of attempts: don't sleep, just fail
        # Server hint beats our own backoff, but never exceed the deadline.
        hint = resp.headers.get('Retry-After')
        sleep = parse_retry_after(hint)
        if sleep is None:
            sleep = backoff_delay(attempt)
        sleep = min(sleep, remaining)
        print(f'  attempt {attempt}: status {resp.status}'
              + (f' (Retry-After={hint})' if hint else '') + f' → sleep {sleep:.2f}s')
        time.sleep(sleep)
    raise RuntimeError(f'gave up after {max_attempts} attempts')


In [ ]:
# Demo A: server returns 503 with Retry-After: 1, then 200.
random.seed(5)
server = FakeServer(script=['retry-after:1', 200])
resp, n = call_with_retry(lambda: server.post('/charge'))
print(f'✅ success on attempt {n}: {resp.body}')


In [ ]:
# Demo B: server returns 400 immediately — must NOT retry.
random.seed(6)
server = FakeServer(script=[400])
try:
    call_with_retry(lambda: server.post('/charge'))
except RuntimeError as e:
    print(f'❌ correctly did NOT retry: {e}')


In [ ]:
# Demo C: server keeps dropping the connection — deadline saves us.
random.seed(7)
server = FakeServer(script=['conn-drop'] * 20)
try:
    call_with_retry(lambda: server.post('/charge'), deadline_s=1.0, max_attempts=50)
except (TimeoutError, ConnectionError) as e:
    print(f'⌛ bailed out: {type(e).__name__}: {e}')


## 3. 🔑 Idempotency keys — safe retries for `POST`

If your client retries a `POST /charge`, you might charge the customer
twice. The fix: send a unique **idempotency key** (UUID) per *logical*
request. Servers deduplicate by that key, so retrying with the same key
is safe.

This is how Stripe, GitHub, and most modern APIs handle writes.

In [ ]:
random.seed(8)
server = FakeServer(script=[200, 200])  # server would 'succeed' twice!
key = str(uuid.uuid4())

r1 = server.post('/charge', idempotency_key=key)
r2 = server.post('/charge', idempotency_key=key)  # retried with SAME key
print(f'first:  {r1}')
print(f'second: {r2}  ← deduplicated, no double charge')


## 4. 💰 Retry budgets — protect downstream when everything is failing

If the downstream is healthy, retries are rare. If it's **completely** down,
retries *triple or quadruple* your load at the worst possible moment.

A **retry budget** caps the *ratio* of retries to regular calls. E.g.
"allow up to 10% of all calls to be retries." When the budget is exhausted,
further failures bubble up immediately — no retry attempted.

This is what Google's SRE book and Envoy implement.

In [ ]:
from collections import deque

class RetryBudget:
    """Sliding-window retry budget. Allows at most `ratio` retries per call."""
    def __init__(self, ratio=0.1, window=100, min_calls=10):
        self.ratio = ratio
        self.window = window
        self.min_calls = min_calls
        self.events = deque()  # each entry: ('call' | 'retry')

    def _trim(self):
        while len(self.events) > self.window:
            self.events.popleft()

    def record_call(self):
        self.events.append('call')
        self._trim()

    def can_retry(self):
        calls = sum(1 for e in self.events if e == 'call')
        retries = sum(1 for e in self.events if e == 'retry')
        if calls < self.min_calls:
            return True  # not enough data yet
        return retries < calls * self.ratio

    def record_retry(self):
        self.events.append('retry')
        self._trim()

# Simulate: 100 calls, 80 of which fail and try to retry. Budget = 10%.
budget = RetryBudget(ratio=0.1, window=200, min_calls=5)
allowed, denied = 0, 0
for i in range(100):
    budget.record_call()
    failed = i < 80  # first 80 calls fail
    if failed:
        if budget.can_retry():
            budget.record_retry()
            allowed += 1
        else:
            denied += 1

print(f'retries allowed: {allowed}')
print(f'retries denied by budget: {denied}  ← these would have piled on a failing service')


## 5. Per-attempt timeout — don't let one hung call eat your deadline

An **overall deadline** (section 2) bounds the total time. But what if a single request
hangs forever? Without a **per-attempt timeout**, a hung call silently burns your entire
deadline, and you never even get to retry.

Rule: **every** outbound call needs a per-attempt timeout. Always.

In [ ]:
import threading

class SlowServer:
    """Simulates a server that never responds on the first call, then works."""
    def __init__(self):
        self.calls = 0
    def get(self):
        self.calls += 1
        if self.calls == 1:
            time.sleep(10)   # hangs! we'd never come back without a timeout
        return Response(200, 'ok')

def call_with_timeout(fn, timeout_s):
    """Run `fn` in a thread; raise TimeoutError if it doesn't finish in time.
    In real code, pass `timeout=` to your HTTP client (requests/httpx/urllib)."""
    result = {}
    def runner():
        try: result['ok'] = fn()
        except Exception as e: result['err'] = e
    t = threading.Thread(target=runner, daemon=True)
    t.start()
    t.join(timeout_s)
    if t.is_alive():
        raise TimeoutError(f'per-attempt timeout after {timeout_s}s')
    if 'err' in result:
        raise result['err']
    return result['ok']

server = SlowServer()
start = time.monotonic()
for attempt in range(1, 4):
    try:
        resp = call_with_timeout(server.get, timeout_s=0.3)
        print(f'attempt {attempt}: OK {resp.body}  (elapsed {time.monotonic()-start:.2f}s)')
        break
    except TimeoutError as e:
        print(f'attempt {attempt}: TIMEOUT {e}  (elapsed {time.monotonic()-start:.2f}s) -> retry')


Without the per-attempt timeout the first call would have blocked 10 seconds —
far beyond any reasonable deadline. With it, we bail out in 300 ms and retry successfully.

> In real code you rarely need to roll your own timeout thread — every modern HTTP client
> supports a timeout option: `requests.get(url, timeout=0.3)`, `httpx.get(url, timeout=0.3)`,
> `urllib.request.urlopen(..., timeout=0.3)`.

## 6. Jitter variants — full vs equal vs decorrelated

We've been using "full jitter" (`uniform(0, delay)`). There are three common variants,
all discussed in AWS's *"Exponential Backoff And Jitter"* post:

| Variant | Formula | Characteristic |
|---|---|---|
| **No jitter** | `base * 2^(a-1)` | Deterministic — herd-prone |
| **Equal jitter** | `delay/2 + uniform(0, delay/2)` | Always at least half the nominal wait |
| **Full jitter** | `uniform(0, delay)` | Max spread — AWS recommends this for most APIs |
| **Decorrelated jitter** | `uniform(base, prev_delay * 3)` | Self-adapting; used by AWS SDKs |

Let's implement and compare them:

In [ ]:
def no_jitter(attempt, base=0.1, factor=2.0, cap=2.0, **_):
    return min(cap, base * (factor ** (attempt - 1)))

def equal_jitter(attempt, base=0.1, factor=2.0, cap=2.0, **_):
    d = min(cap, base * (factor ** (attempt - 1)))
    return d / 2 + random.uniform(0, d / 2)

def full_jitter(attempt, base=0.1, factor=2.0, cap=2.0, **_):
    d = min(cap, base * (factor ** (attempt - 1)))
    return random.uniform(0, d)

def decorrelated_jitter(attempt, base=0.1, cap=2.0, prev=None, **_):
    prev = prev if prev is not None else base
    return min(cap, random.uniform(base, prev * 3))

random.seed(0)
print(f'{"attempt":>7}  {"no-jitter":>10}  {"equal":>10}  {"full":>10}  {"decorrelated":>12}')
prev = 0.1
for a in range(1, 7):
    nj = no_jitter(a)
    ej = equal_jitter(a)
    fj = full_jitter(a)
    dj = decorrelated_jitter(a, prev=prev); prev = dj
    print(f'{a:>7}  {nj:>10.3f}  {ej:>10.3f}  {fj:>10.3f}  {dj:>12.3f}')


**How to choose:**

- **Default** → *full jitter*. Simple, robust, and usually the best spread.
- **Need a minimum wait** (e.g. to avoid super-tight hot loops) → *equal jitter*.
- **Want retries to adapt to recent backoff history** → *decorrelated jitter* — used by
  the AWS SDKs' `StandardRetryStrategy` and recommended for API Gateway.

## 7. 🔗 Retry + Circuit Breaker + Timeout — how they compose

Retries are one ingredient. In production you pair them with:

- **Per-attempt timeout** → don't wait forever for a single response.
- **Overall deadline** → the client's total budget (what we used above).
- **Circuit breaker** → when failure rate is high, stop sending any calls
  for a while. See `05-microservices/circuit-breaker`.
- **Bulkhead** → limit concurrent retries so they can't exhaust your
  thread/connection pool. See `05-microservices/bulkhead`.

Order of checks in a request pipeline:

```
caller
  └── retry (with jitter + budget + deadline)
        └── circuit breaker (fail fast if open)
              └── bulkhead (limit concurrency)
                    └── per-attempt timeout
                          └── actual RPC
```

## 8. 🧭 When to retry — and when **not** to

Retrying is not a default-on setting. It trades a little extra load for a lot of
extra success — but only when the failure is actually transient.

### ✅ Retry when
- The error is **transient**: connection reset, DNS blip, `503`, `504`, leader election,
  cold start, a brief lock contention.
- The operation is **idempotent** — or you carry an **idempotency key** (section 3).
- You have a **deadline** the caller still cares about. Retrying after the user gave up
  is pure waste (and extra load).
- The dependency has spare capacity. Retries are only free when someone can serve them.

### 🚫 Do **not** retry when
- The error is **terminal**: `400`, `401`, `403`, `404`, `422`. The request is wrong;
  the tenth attempt will be just as wrong.
- The write is **not idempotent** and you have no key — a retried `POST /charge`
  is a second charge, not a second chance.
- The dependency is **saturated**, not flaky. Retrying an overloaded service is how a
  brownout becomes an outage. Use a **circuit breaker** + **load shedding** instead.
- You are **out of budget** — no time left on the deadline, or the retry budget
  (section 4) is exhausted.
- The call is **long-running or expensive** (a 30 s report, an LLM call). One retry can
  double your bill and your tail latency; prefer an async job + status polling.
- You are retrying **at more than one layer**. Client retries × sidecar retries ×
  library retries multiply: 3 × 3 × 3 = 27 requests for one logical call. Pick **one**
  layer to own retries and turn the others off.


## ✅ Production retry checklist

- [ ] Only retry **idempotent** operations (or use idempotency keys).
- [ ] **Classify** errors — never retry 4xx (except 408/425/429).
- [ ] **Exponential backoff + jitter** — never immediate, never fixed.
- [ ] **Cap** the per-attempt delay (e.g. 5–30 seconds).
- [ ] **Max attempts** AND **overall deadline** — both, not just one.
- [ ] **Honor `Retry-After`** from 429/503 responses.
- [ ] **Retry budget** to prevent storms when everything fails.
- [ ] **Circuit breaker** in front of retries for fail-fast behavior.
- [ ] **Per-attempt timeout** — a hung request blocks retry logic.
- [ ] **Log & emit metrics** — retries are a leading indicator of problems.